# Mock Pipeline Test Runner
Smoke-tests the full walk-forward pipeline end-to-end using the mock Synthea profile (1000 patients, 3 years).  
No modelling — stops after one walk-forward month.  

**Cells 1–10:** Initial creation (Synthea → BQ raw → slim → dictionaries → helpers → index_stay)  
**Cell 11:** Compute watermark from MAX(stop) − 1 year  
**Cell 12:** Walk-forward: one month via WalkForwardOrchestrator  
**Cell 13:** Final status check  

In [ ]:
# Cell 0: Setup & imports
import sys
import json
import shutil
import calendar
from datetime import date
from pathlib import Path

import pandas as pd

# Notebook lives at scripts/ — project root is one level up
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
# pipeline modules use both `from pipeline.x import` AND `from src.utils.x import`
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

from pipeline.synthea_runner import SyntheaRunner
from pipeline.synthea_segmenter import SyntheaSegmenter
from pipeline.bq_loader import BigQueryLoader
from pipeline.bq_transformer import BigQueryTransformer
from pipeline.dictionary_builder import DictionaryBuilder
from pipeline.walk_forward import WalkForwardOrchestrator
from pipeline.preprocessing import DataPreprocessor
from pipeline.model_config_manager import ModelConfigManager
from pipeline.model_registry import ModelRegistry
from pipeline.hyperparameter_tuner import HyperparameterTuner
from pipeline.evaluator import Evaluator
from pipeline.cost_reducer import CostReducer

# Shared config paths (used across all cells)
synthea_config_path = str(project_root / "config" / "synthea_config.json")
config_path_bq      = str(project_root / "config" / "bigquery_config.json")
recipe_path         = str(project_root / "config" / "bigquery_recipes.json")
io_config_path      = str(project_root / "config" / "dictionary_io_config.json")
watermark_path      = str(project_root / "config" / "watermark.json")
model_config_path   = str(project_root / "config" / "model_config.json")
cost_config_path    = str(project_root / "config" / "cost_config.json")
checks_dir          = str(project_root / "sql" / "checks")

print(f"project_root: {project_root}")

In [ ]:
# Cell 1: Archive existing local files before overwriting
today = date.today().strftime("%Y%m%d")
archive_root = project_root / "data" / "archive" / today

FILES_TO_ARCHIVE = [
    # Synthea raw CSVs
    "data/raw/Synthea/mock/patients.csv",
    "data/raw/Synthea/mock/encounters.csv",
    "data/raw/Synthea/mock/careplans.csv",
    "data/raw/Synthea/mock/claims.csv",
    "data/raw/Synthea/mock/conditions.csv",
    "data/raw/Synthea/mock/medications.csv",
    "data/raw/Synthea/mock/organizations.csv",
    "data/raw/Synthea/mock/procedures.csv",
    # BQ fetch caches (data_path in dictionary_io_config.json)
    "data/raw/dictionaries/unique_diagnoses.csv",
    "data/raw/dictionaries/unique_procedures.csv",
    "data/raw/diagnoses_per_stays.csv",
    "data/raw/diagnoses_and_careplans.csv",
    "data/raw/diagnoses_and_following.csv",
    # SNOMED classification state
    "data/intermediate/diagnosess_snomed_state.json",
    "data/intermediate/procedures_snomed_state.json",
    # Processed dictionaries (write_path in dictionary_io_config.json)
    "data/processed/dictionaries/diagnoses_dictionary.csv",
    "data/processed/dictionaries/procedures_dictionary.csv",
    # Watermark
    "config/watermark.json",
]

archived, skipped = [], []
for rel in FILES_TO_ARCHIVE:
    src = project_root / rel
    if src.exists():
        dst = archive_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        archived.append(rel)
    else:
        skipped.append(rel)

print(f"Archive path: {archive_root}")
print(f"Archived ({len(archived)}):")
for f in archived:
    print(f"  {f}")
if skipped:
    print(f"\nNot found / skipped ({len(skipped)}):")
    for f in skipped:
        print(f"  {f}")

In [ ]:
# Cell 2: Run Synthea (mock: 1000 patients, 3 years, seed=100)
runner, run_params = SyntheaRunner.from_profile(synthea_config_path, "mock")
runner.run(**run_params)
print("Synthea done.")

In [ ]:
# Cell 3: Segment Synthea CSVs into base + monthly files
# RESEGMENT = True  → re-runs segmentation (required after a fresh Synthea run)
# RESEGMENT = False → skips segmentation, derives window dates from existing encounters.csv
RESEGMENT = False
segmenter = SyntheaSegmenter(config_path_bq, "mock")
if RESEGMENT:
    segmenter.segment(overwrite=True)
else:
    segmenter.derive_window_from_existing()
segmenter.write_watermark(watermark_path)
print(f"simulation_start    : {segmenter.simulation_start}")
print(f"base_cutoff_date    : {segmenter.base_cutoff_date}")
print(f"simulation_end_date : {segmenter.simulation_end_date}")

In [ ]:
# Cell 4: Load base segment CSVs into BigQuery raw_data dataset
# Uses *_base.csv files from segmented_path (pre-simulation records only)
bq_loader, profile_cfg = BigQueryLoader.from_profile(config_path_bq, "mock")
bq_loader.load_base_segment()
print("Base segment CSVs loaded to BQ.")

In [ ]:
# Cell 5: Create slim tables + build dictionaries
# Dates come from SyntheaSegmenter — no BQ query needed
base_cutoff_iso = segmenter.base_cutoff_date.isoformat()

_last_day     = calendar.monthrange(segmenter.simulation_start.year, segmenter.simulation_start.month)[1]
next_end_date = date(segmenter.simulation_start.year, segmenter.simulation_start.month, _last_day).isoformat()

print(f"base_cutoff_iso : {base_cutoff_iso}")
print(f"next_end_date   : {next_end_date}")

transformer, _ = BigQueryTransformer.from_profile(config_path_bq)

# Create base slim tables (pre-simulation records only, filtered to base_cutoff_date)
transformer.run_query_sequence(recipe_path, 0, str(project_root), end_date=base_cutoff_iso)
print("Slim tables created.")

# Build dictionaries from base data
builder = DictionaryBuilder(transformer=transformer, io_config_path=io_config_path)
builder.build_diagnoses_dictionary(end_date=base_cutoff_iso)
builder.build_procedures_dictionary(end_date=base_cutoff_iso)
builder.build_main_diagnoses(end_date=base_cutoff_iso)
print("Dictionaries built locally.")

In [ ]:
# Cell 6: Load dictionaries to BQ
bq_loader.load_dictionaries()
print("Dictionaries loaded to BQ.")

In [ ]:
# Cell 7: Build careplans_related_diagnoses locally + load to BQ
builder.build_careplans_related_diagnoses()
bq_loader.load_careplans()
print("Careplans loaded to BQ.")

In [ ]:
# Cell 8: Create helper tables (recipe index 1) + run all 3 sanity checks
transformer.run_query_sequence(recipe_path, 1, str(project_root))
print("Helper tables created.")

transformer.run_helper_clinical_sanity_checks(checks_dir)
transformer.run_helper_cost_sanity_checks(checks_dir)
transformer.run_helper_utilization_sanity_checks(checks_dir)
print("All sanity checks passed.")

In [ ]:
# Cell 9: Build related_diagnoses locally + load to BQ
builder.build_related_diagnoses()
bq_loader.load_related_diagnoses()
print("related_diagnoses loaded to BQ.")

In [ ]:
# Cell 10: Create index_stay (recipe index 2)
transformer.run_query_sequence(recipe_path, 2, str(project_root))
print("index_stay created.")

In [ ]:
# Cell 12: Walk-forward — run one month
orch = WalkForwardOrchestrator(
    transformer=transformer,
    dict_builder=builder,
    recipe_path=recipe_path,
    project_root=str(project_root),
    watermark_path=watermark_path,
)

processed_end_date = orch.run_next_month()
print(f"\nWalk-forward month complete: {processed_end_date}")

In [ ]:
# Cell 12: Walk-forward — base predictions + 2 simulation months
import json as _json

preprocessor = DataPreprocessor.from_config(model_config_path)
registry     = ModelRegistry.from_config(model_config_path)
cfg_mgr      = registry.config_mgr
tuner        = HyperparameterTuner(config_mgr=cfg_mgr, target_col="readmit_30d", cost_config_path=cost_config_path)
evaluator    = Evaluator(registry=registry, cfg_mgr=cfg_mgr)
cost_reducer = CostReducer.from_config(cost_config_path)

orch = WalkForwardOrchestrator(
    transformer=transformer,
    dict_builder=builder,
    loader=bq_loader,
    recipe_path=recipe_path,
    project_root=str(project_root),
    watermark_path=watermark_path,
    preprocessor=preprocessor,
    registry=registry,
    tuner=tuner,
    evaluator=evaluator,
    cost_reducer=cost_reducer,
    index_stay_sql_path=str(project_root / "sql" / "20_index_stay_selection.sql"),
)

# Build predictions for base dataset (first_run path fires — no predictions files exist)
base_cutoff_date = segmenter.base_cutoff_date.isoformat()
orch.fit_and_evaluate(base_cutoff_date)
print(f"Base predictions built for {base_cutoff_date}")

# Bootstrap prior-month staging tables needed by first run_month
with open(watermark_path) as _f:
    _wm = _json.load(_f)
orch.bootstrap_prior_month_staging(_wm["next_end_date"])

# Run 2 simulation months
for i in range(2):
    processed = orch.run_next_month()
    print(f"Month {i+1} complete: {processed}")

print("2-month simulation complete.")